# Step 2: Explore and Prepare Your Dataset

This notebook loads the NL-to-regex pairs, inspects the data, cleans it, validates regex syntax, splits into train/val/test, and defines a tokenization strategy.

In [1]:
from pathlib import Path
import re
import pandas as pd
from sklearn.model_selection import train_test_split

## 2.1 Load and Inspect the Dataset

This dataset is stored as paired text files. We load NL descriptions from `src.txt` and regex from `targ.txt`, then inspect size, schema, and basic distributions.

In [2]:
# Set dataset folder here
DATASET_DIR = Path("datasets/NL-RX-Turk")

src_path = DATASET_DIR / "src.txt"
targ_path = DATASET_DIR / "targ.txt"

if not src_path.exists() or not targ_path.exists():
    raise FileNotFoundError(f"Missing dataset files in {DATASET_DIR}")

# Load paired lines
with src_path.open("r", encoding="utf-8") as f:
    src_lines = [line.rstrip("\n") for line in f]

with targ_path.open("r", encoding="utf-8") as f:
    targ_lines = [line.rstrip("\n") for line in f]

if len(src_lines) != len(targ_lines):
    raise ValueError(f"Line count mismatch: src={len(src_lines)} targ={len(targ_lines)}")

df = pd.DataFrame({"nl": src_lines, "regex": targ_lines})

print("Rows:", len(df))
print(df.head(5))

Rows: 10000
                                                  nl  \
0  Lines that have words containing a capital let...   
1  Lends that end with a character and a string '...   
2     lines that have either a letter or a character   
3  Items with a numeral preceding a letter or sma...   
4       lines with the string 'dog' at least 2 times   

                             regex  
0       \b([A-Z])&([AEIOUaeiou])\b  
1               ((.*)(.)).*(dog).*  
2               .*([A-Za-z])|(.).*  
3  ([0-9]).*(([A-Za-z])&([a-z])).*  
4                    .*(dog){2,}.*  


In [3]:
# Format check
print("Source format:", src_path.suffix)
print("Target format:", targ_path.suffix)

Source format: .txt
Target format: .txt


In [4]:
# Basic inspection
print("Nulls:\n", df.isna().sum())
print("Empty strings:\n", (df["nl"].str.len() == 0).sum(), (df["regex"].str.len() == 0).sum())
print("Duplicate pairs:", df.duplicated(subset=["nl", "regex"]).sum())

# Length distributions (simple summary)
len_stats = df.assign(nl_len=df["nl"].str.len(), regex_len=df["regex"].str.len())
print(len_stats[["nl_len", "regex_len"]].describe())

Nulls:
 nl       0
regex    0
dtype: int64
Empty strings:
 0 0
Duplicate pairs: 1
             nl_len     regex_len
count  10000.000000  10000.000000
mean      60.556800     25.989900
std       16.142423      7.421799
min        9.000000      1.000000
25%       49.000000     21.000000
50%       60.000000     25.000000
75%       71.000000     31.000000
max      132.000000     54.000000


In [5]:
def regex_type(pattern: str) -> str:
    # Heuristic grouping to spot rare regex types
    if pattern.startswith("^") and pattern.endswith("$"):
        return "anchored"
    if "\\d" in pattern:
        return "digit_class"
    if "[" in pattern and "]" in pattern:
        return "char_class"
    if "|" in pattern:
        return "alternation"
    if "*" in pattern or "+" in pattern or "?" in pattern:
        return "quantifier"
    return "other"

regex_types = df["regex"].map(regex_type)
print(regex_types.value_counts())

regex
char_class     8803
quantifier      768
alternation     325
other           104
Name: count, dtype: int64


## 2.2 Clean the Data

We remove duplicates and nulls, normalize whitespace in NL descriptions, and validate regex syntax with Python's `re` module.

In [6]:
def normalize_whitespace(text: str) -> str:
    # Collapse runs of whitespace to a single space
    return re.sub(r"\s+", " ", text).strip()

clean_df = df.copy()

# Drop nulls or empty strings
clean_df = clean_df.dropna(subset=["nl", "regex"]).copy()
clean_df = clean_df[(clean_df["nl"].str.len() > 0) & (clean_df["regex"].str.len() > 0)]

# Normalize NL whitespace
clean_df["nl"] = clean_df["nl"].map(normalize_whitespace)

# Remove duplicate pairs
clean_df = clean_df.drop_duplicates(subset=["nl", "regex"]).reset_index(drop=True)

# Validate regex syntax
invalid_regex = []
for i, pattern in enumerate(clean_df["regex"]):
    try:
        re.compile(pattern)
    except re.error:
        invalid_regex.append(i)

print("Clean rows:", len(clean_df))
print("Invalid regex patterns:", len(invalid_regex))

Clean rows: 9999
Invalid regex patterns: 0


In [7]:
# Drop invalid regex rows if any
if invalid_regex:
    clean_df = clean_df.drop(index=invalid_regex).reset_index(drop=True)

print("Final rows after dropping invalid regex:", len(clean_df))

Final rows after dropping invalid regex: 9999


## 2.3 Split the Dataset

We split into Train / Validation / Test using 80/10/10. If regex types are rare, we attempt stratified splitting based on a simple regex-type heuristic.

In [8]:
clean_df = clean_df.reset_index(drop=True)
regex_types_clean = clean_df["regex"].map(regex_type)

# Check if stratification is feasible
type_counts = regex_types_clean.value_counts()
can_stratify = (type_counts.min() >= 2)

rng = 42

if can_stratify:
    try:
        train_df, temp_df = train_test_split(
            clean_df,
            test_size=0.2,
            random_state=rng,
            stratify=regex_types_clean,
        )
        temp_types = temp_df["regex"].map(regex_type)
        train_df, val_df = train_test_split(
            train_df,
            test_size=0.1111111111,
            random_state=rng,
            stratify=train_df["regex"].map(regex_type),
        )
        val_df, test_df = train_test_split(
            temp_df,
            test_size=0.5,
            random_state=rng,
            stratify=temp_types,
        )
    except ValueError:
        can_stratify = False

if not can_stratify:
    train_df, temp_df = train_test_split(
        clean_df,
        test_size=0.2,
        random_state=rng,
        shuffle=True,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=rng,
        shuffle=True,
    )

print("Train/Val/Test sizes:", len(train_df), len(val_df), len(test_df))

# Persist split indices for reproducibility
splits_dir = Path("datasets/splits")
splits_dir.mkdir(parents=True, exist_ok=True)
train_df.to_csv(splits_dir / "train.csv", index=False)
val_df.to_csv(splits_dir / "val.csv", index=False)
test_df.to_csv(splits_dir / "test.csv", index=False)

Train/Val/Test sizes: 7110 1000 1000


## 2.4 Define Tokenization Strategy

- NL input: use a pre-trained tokenizer (BPE or WordPiece) from a Seq2Seq model.
- Regex output: use character-level tokenization to preserve exact symbols like `*`, `+`, `\\d`, and `[]`.

In [9]:
def tokenize_regex_chars(pattern: str) -> list[str]:
    # Character-level tokens preserve regex symbols exactly
    return list(pattern)

sample_nl = clean_df["nl"].iloc[0]
sample_regex = clean_df["regex"].iloc[0]

print("Sample NL:", sample_nl)
print("Sample regex:", sample_regex)
print("Regex char tokens:", tokenize_regex_chars(sample_regex)[:50])

# Optional: pre-trained tokenizer for NL (requires transformers)
try:
    from transformers import AutoTokenizer  # type: ignore

    tokenizer = AutoTokenizer.from_pretrained("t5-small")
    nl_tokens = tokenizer.tokenize(sample_nl)
    print("NL tokens (t5-small):", nl_tokens[:50])
except Exception as exc:
    print("Transformers not available, install if needed:", exc)

Sample NL: Lines that have words containing a capital letter as well as a vowel
Sample regex: \b([A-Z])&([AEIOUaeiou])\b
Regex char tokens: ['\\', 'b', '(', '[', 'A', '-', 'Z', ']', ')', '&', '(', '[', 'A', 'E', 'I', 'O', 'U', 'a', 'e', 'i', 'o', 'u', ']', ')', '\\', 'b']


C:\Users\chand\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NL tokens (t5-small): ['▁Line', 's', '▁that', '▁have', '▁words', '▁', 'containing', '▁', 'a', '▁capital', '▁letter', '▁as', '▁well', '▁as', '▁', 'a', '▁vow', 'e', 'l']


## Step 3: Choose a Pre-Trained Model

We will use `Salesforce/codet5-base` for NL-to-regex generation.

In [10]:
MODEL_NAME = "Salesforce/codet5-base"
print("Model:", MODEL_NAME)

Model: Salesforce/codet5-base


## Step 4: Set Up the Environment

Run these once in your terminal:

```bash
pip install transformers datasets torch scikit-learn
pip install evaluate rouge-score sacrebleu sentencepiece
```

## Step 5: Tokenize and Format Data for the Model

We load the tokenizer, define a tokenization function, and map it over the train/val/test splits.

In [11]:
from datasets import Dataset
from transformers import AutoTokenizer

MAX_INPUT_LEN = 128
MAX_TARGET_LEN = 64

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
except Exception:
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            use_fast=True,
            additional_special_tokens=[],
        )
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

print("Tokenizer vocab size:", tokenizer.vocab_size)

Tokenizer vocab size: 32100


In [12]:
splits_dir = Path("datasets/splits")
train_csv = splits_dir / "train.csv"
val_csv = splits_dir / "val.csv"
test_csv = splits_dir / "test.csv"

if not (train_csv.exists() and val_csv.exists() and test_csv.exists()):
    raise FileNotFoundError(
        "Split files not found. Run the Step 2 split cell first to create datasets/splits/*.csv"
    )

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print("Loaded splits:", len(train_df), len(val_df), len(test_df))

Loaded splits: 7110 1000 1000


In [13]:
def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["nl"],
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["regex"],
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["nl", "regex"])
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=["nl", "regex"])
test_tok = test_ds.map(tokenize_batch, batched=True, remove_columns=["nl", "regex"])

print(train_tok[0].keys())
print("Tokenized sizes:", len(train_tok), len(val_tok), len(test_tok))

Map: 100%|██████████| 1000/1000 [00:00<00:00, 13498.66 examples/s]

dict_keys(['input_ids', 'attention_mask', 'labels'])
Tokenized sizes: 7110 1000 1000


## Step 6: Load the Pre-Trained Model

We load the Seq2Seq model from Hugging Face. It already includes cross-attention between encoder and decoder, which we will visualize later.

In [14]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Loaded model:", MODEL_NAME)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 40843.41it/s]


Loaded model: Salesforce/codet5-base


## Step 7: Train the Model (Fine-Tuning)

We use `Seq2SeqTrainingArguments` and `Seq2SeqTrainer` with BLEU/ROUGE metrics. Monitor training and validation loss for overfitting and keep the best model based on validation loss.

In [16]:
import evaluate
import numpy as np
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    rouge_scores = rouge.compute(predictions=decoded_preds, references=decoded_labels)
    bleu_scores = bleu.compute(predictions=decoded_preds, references=[[r] for r in decoded_labels])

    return {
        "rouge1": rouge_scores["rouge1"],
        "rougeL": rouge_scores["rougeL"],
        "sacrebleu": bleu_scores["score"],
    }

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="outputs/codet5-base-nl2regex",
    num_train_epochs=10,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=50,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Uncomment to train
# trainer.train()

## Step 8: Add and Visualize Attention

We extract encoder, decoder, and cross-attention and visualize cross-attention as a heatmap. Install `bertviz` for interactive multi-head views if desired.

In [17]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: pip install bertviz

def get_cross_attention(nl_text: str):
    model.eval()
    inputs = tokenizer(nl_text, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
        )

        # Use generated ids as decoder inputs (shifted for attention alignment)
        decoder_input_ids = gen_ids[:, :-1]
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            decoder_input_ids=decoder_input_ids,
            output_attentions=True,
            return_dict=True,
        )

    cross_attentions = outputs.cross_attentions  # tuple of layers
    # Stack: layers x batch x heads x tgt_len x src_len
    cross = torch.stack([c[0] for c in cross_attentions], dim=0)
    cross = cross.mean(dim=0).mean(dim=0)  # avg layers, avg heads => tgt_len x src_len

    src_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tgt_tokens = tokenizer.convert_ids_to_tokens(decoder_input_ids[0])

    return cross.cpu().numpy(), src_tokens, tgt_tokens


def plot_cross_attention(nl_text: str):
    cross, src_tokens, tgt_tokens = get_cross_attention(nl_text)
    plt.figure(figsize=(12, 6))
    sns.heatmap(cross, xticklabels=src_tokens, yticklabels=tgt_tokens, cmap="viridis")
    plt.xlabel("Input NL tokens")
    plt.ylabel("Generated regex tokens")
    plt.title("Cross-Attention Heatmap")
    plt.tight_layout()
    plt.show()

# Example usage (run after training)
# plot_cross_attention("match strings that start with a digit")

## Step 9: Generate Predictions on Test Set

We run the trained model on the test set with beam search and decode predictions back to strings.

In [18]:
# Generate predictions on test set (run after training)
# preds = trainer.predict(test_tok)
# pred_ids = preds.predictions
# if isinstance(pred_ids, tuple):
#     pred_ids = pred_ids[0]
#
# pred_text = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
# ref_text = test_df["regex"].tolist()
#
# print("Sample prediction:", pred_text[0] if pred_text else "")
# print("Sample reference:", ref_text[0] if ref_text else "")

## Step 10: Compute BLEU, ROUGE, and Exact Match (EM)

We compute BLEU (including BLEU-1/2/4), ROUGE-1/2/L, and exact match accuracy on the test set.

In [19]:
# Compute BLEU, ROUGE, and Exact Match (run after generating pred_text/ref_text)
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
#
# bleu = evaluate.load("sacrebleu")
# rouge = evaluate.load("rouge")
#
# bleu_scores = bleu.compute(predictions=pred_text, references=[[r] for r in ref_text])
# rouge_scores = rouge.compute(predictions=pred_text, references=ref_text)
#
# # Approximate BLEU-1/2/4 via NLTK for per-sentence averaging
# smooth = SmoothingFunction().method1
# bleu1 = np.mean([sentence_bleu([r.split()], p.split(), weights=(1, 0, 0, 0), smoothing_function=smooth)
#                  for p, r in zip(pred_text, ref_text)])
# bleu2 = np.mean([sentence_bleu([r.split()], p.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
#                  for p, r in zip(pred_text, ref_text)])
# bleu4 = np.mean([sentence_bleu([r.split()], p.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
#                  for p, r in zip(pred_text, ref_text)])
#
# exact_match = np.mean([p == r for p, r in zip(pred_text, ref_text)])
#
# print("SacreBLEU:", bleu_scores)
# print("BLEU-1:", bleu1, "BLEU-2:", bleu2, "BLEU-4:", bleu4)
# print("ROUGE:", rouge_scores)
# print("Exact Match:", exact_match)

## Step 11: Analyze Results and Iterate

We compare performance by regex complexity, inspect near-miss errors, and categorize failure modes to guide improvements.

In [20]:
# Analysis helpers (run after pred_text/ref_text exist)
#
# def categorize_failure(pred: str, ref: str) -> str:
#     if pred == ref:
#         return "exact"
#     if "\\d" in ref and "\\d" not in pred:
#         return "missing_digit_class"
#     if ref.startswith("^") and not pred.startswith("^"):
#         return "missing_anchor_start"
#     if ref.endswith("$") and not pred.endswith("$"):
#         return "missing_anchor_end"
#     if any(q in ref for q in ["*", "+", "?"]) and not any(q in pred for q in ["*", "+", "?"]):
#         return "missing_quantifier"
#     if "\\" in ref and "\\" not in pred:
#         return "missing_escape"
#     return "other"
#
# # Compare by regex complexity
# test_types = test_df["regex"].map(regex_type)
# df_eval = pd.DataFrame({"pred": pred_text, "ref": ref_text, "type": test_types})
# df_eval["exact"] = df_eval["pred"] == df_eval["ref"]
#
# print("Exact match by type:")
# print(df_eval.groupby("type")["exact"].mean().sort_values(ascending=False))
#
# # Near-miss errors: high token overlap but wrong regex
# def token_overlap(a: str, b: str) -> float:
#     a_set, b_set = set(a), set(b)
#     return len(a_set & b_set) / max(1, len(a_set | b_set))
#
# df_eval["overlap"] = [token_overlap(p, r) for p, r in zip(pred_text, ref_text)]
# near_miss = df_eval[(df_eval["exact"] == False) & (df_eval["overlap"] >= 0.7)]
# print("Near-miss examples:")
# print(near_miss.head(5))
#
# # Failure type breakdown
# df_eval["failure_type"] = [categorize_failure(p, r) for p, r in zip(pred_text, ref_text)]
# print("Failure types:")
# print(df_eval["failure_type"].value_counts())
#
# # Iteration ideas (manual):
# # - Increase epochs or switch to a larger model (t5-large)
# # - Augment data with paraphrases of NL descriptions
# # - Try character-level decoding for regex outputs

## Step 12: Document and Present

Use the following structure for your report or final notebook sections:

### Introduction
- Problem statement and motivation

### Dataset Analysis
- Dataset size, stats, examples, and distribution

### Model Architecture
- Encoder-decoder diagram with attention

### Training Setup
- Hyperparameters and hardware details

### Attention Visualizations
- Heatmaps and qualitative analysis

### Results
- BLEU, ROUGE, Exact Match (EM) table

### Error Analysis
- Failure cases and patterns

### Conclusion
- What worked, what did not, and future improvements